# 🦠 FloodShield AI — Model 2B: Disease-wise Case Count Prediction
**Algorithm:** XGBoost Regressor (one per disease, 6 total)
**Input:** Same leakage-free features as Model 2 (rainfall, temperature, humidity, river level, flood depth, population density, sanitation, water quality, historical cases, season)
**Output:** Predicted case count for EACH of 6 diseases separately — Cholera, Diarrhea, Typhoid, Dengue, Hepatitis A, Skin Disease

### Why this exists
Model 2 predicts overall outbreak risk (Medium/High/Critical) but doesn't say *which* disease. This model answers that — instead of "Dengue is dominant," it gives actual expected case counts per disease, e.g. "Dengue: ~45 cases, Diarrhea: ~30 cases" — more actionable for hospital resource planning.

### How to run:
1. `disease_prediction_dataset_fixed.xlsx` should already be in `FloodShieldAI/data/` on Drive
2. Runtime → Run all


## 1️⃣ Setup

In [ ]:
!pip install xgboost scikit-learn pandas openpyxl -q

from google.colab import drive
drive.mount('/content/drive')


## 2️⃣ Load Dataset

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/FloodShieldAI/data/disease_prediction_dataset_fixed.xlsx'

df = pd.read_excel(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 3️⃣ Feature Engineering
Same leakage-free features as Model 2 — `future_cases` excluded (that was used to build Model 2's target, using it here would be leakage for the same reason).


In [ ]:
season_dummies = pd.get_dummies(df['season'], prefix='season')

feature_cols_base = ['rainfall_mm', 'temperature_c', 'humidity_percent', 'river_level_m',
                      'flood_depth_cm', 'population_density', 'sanitation_score',
                      'water_quality_index', 'previous_week_cases', 'previous_two_week_cases']

X = pd.concat([df[feature_cols_base], season_dummies], axis=1)
feature_cols = X.columns.tolist()

disease_targets = ['cholera_cases', 'diarrhea_cases', 'typhoid_cases',
                    'dengue_cases', 'hepatitis_a_cases', 'skin_disease_cases']
y = df[disease_targets]

print("Features:", feature_cols)
print("Targets:", disease_targets)
print(y.describe())


## 4️⃣ Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 5️⃣ Train One XGBoost Regressor Per Disease
A separate model per disease (rather than one multi-output model) keeps each disease's prediction independently tunable and easy to explain — consistent with how the rest of this project favors explainable, per-purpose models over a single opaque one.


In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

disease_models = {}
results = {}

for disease in disease_targets:
    model = XGBRegressor(n_estimators=250, max_depth=6, learning_rate=0.08, random_state=42)
    model.fit(X_train, y_train[disease])

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test[disease], preds)
    r2 = r2_score(y_test[disease], preds)

    disease_models[disease] = model
    results[disease] = {'MAE': round(mae, 2), 'R2': round(r2, 4)}
    print(f"{disease:20s} -> MAE: {mae:.2f} | R²: {r2:.4f}")

print()
results_df = pd.DataFrame(results).T
print(results_df)


## 6️⃣ Feature Importance (per disease, quick comparison)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, disease in enumerate(disease_targets):
    imp = pd.Series(disease_models[disease].feature_importances_, index=feature_cols).sort_values(ascending=False).head(6)
    imp.plot(kind='barh', ax=axes[i])
    axes[i].set_title(disease)
    axes[i].invert_yaxis()

plt.tight_layout()
plt.show()


## 7️⃣ Prediction Function — All 6 Diseases At Once

In [ ]:
def predict_disease_breakdown(rainfall_mm, temperature_c, humidity_percent, river_level_m,
                               flood_depth_cm, population_density, sanitation_score,
                               water_quality_index, previous_week_cases, previous_two_week_cases,
                               season='Monsoon'):
    input_dict = {
        'rainfall_mm': rainfall_mm, 'temperature_c': temperature_c,
        'humidity_percent': humidity_percent, 'river_level_m': river_level_m,
        'flood_depth_cm': flood_depth_cm, 'population_density': population_density,
        'sanitation_score': sanitation_score, 'water_quality_index': water_quality_index,
        'previous_week_cases': previous_week_cases, 'previous_two_week_cases': previous_two_week_cases
    }
    season_cols = [c for c in feature_cols if c.startswith('season_')]
    for col in season_cols:
        season_name = col.replace('season_', '')
        input_dict[col] = 1 if season_name == season else 0

    input_df = pd.DataFrame([input_dict])[feature_cols]

    breakdown = {}
    for disease in disease_targets:
        pred = disease_models[disease].predict(input_df)[0]
        breakdown[disease.replace('_cases', '')] = round(max(0, float(pred)), 1)

    # sort descending so the dominant disease is first
    breakdown_sorted = dict(sorted(breakdown.items(), key=lambda x: x[1], reverse=True))
    dominant = list(breakdown_sorted.keys())[0]

    return {
        'disease_breakdown': breakdown_sorted,
        'dominant_disease': dominant,
        'total_predicted_cases': round(sum(breakdown.values()), 1)
    }

# Example test
print(predict_disease_breakdown(rainfall_mm=220, temperature_c=29, humidity_percent=87,
                                  river_level_m=7.5, flood_depth_cm=55, population_density=2100,
                                  sanitation_score=0.35, water_quality_index=50,
                                  previous_week_cases=210, previous_two_week_cases=190,
                                  season='Monsoon'))


## 8️⃣ Save Models

In [ ]:
import joblib
import os

SAVE_DIR = '/content/drive/MyDrive/FloodShieldAI/models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(disease_models, f'{SAVE_DIR}/disease_breakdown_models.pkl')  # dict of 6 models
joblib.dump(feature_cols, f'{SAVE_DIR}/disease_breakdown_features.pkl')
joblib.dump(disease_targets, f'{SAVE_DIR}/disease_breakdown_target_names.pkl')

print(f"✅ Saved to: {SAVE_DIR}")
print("Files: disease_breakdown_models.pkl, disease_breakdown_features.pkl, disease_breakdown_target_names.pkl")


---
## ✅ What this notebook produces
- 6 separate XGBoost regressors, one per disease, predicting expected case counts
- A single `predict_disease_breakdown()` function returning all 6 at once, sorted by severity, with the dominant disease highlighted
- MAE/R² per disease — honest, per-disease accuracy (some diseases will predict better than others depending on how much signal is in the data; report all 6, don't average them into one misleading number)

## Next step — Backend integration
This becomes a new endpoint, e.g. `POST /predict/disease-breakdown`, loading `disease_breakdown_models.pkl` (a dict of 6 models) alongside the existing Model 2. The frontend's Disease Risk card can then show the breakdown instead of (or alongside) the single risk level.
